## Install Dependencies

In [ ]:
import sys
import os
from pathlib import Path

# Set up paths
HOME = Path.home()
PROJECT_DIR = Path("...../RF-DETR&SAM2/full_noP22")

# Add RF-DETR and SAM2 to path
sys.path.insert(0, str(PROJECT_DIR / "rf-detr"))
sys.path.insert(0, str(PROJECT_DIR / "sam2"))

print(f"✓ Project directory: {PROJECT_DIR}")
print(f"✓ Python paths configured")

## Download SAM2  Checkpoint

In [ ]:
import os
import urllib.request

# Create checkpoints directory
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Download SAM2 base-plus checkpoint
SAM2_CHECKPOINT = CHECKPOINT_DIR / "sam2_hiera_base_plus.pt"
SAM2_CONFIG = "sam2_hiera_b+.yaml"

if not SAM2_CHECKPOINT.exists():
    print("⏳ Downloading SAM2 base-plus checkpoint...")
    url = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_base_plus.pt"
    urllib.request.urlretrieve(url, SAM2_CHECKPOINT)
    print(f"✓ Downloaded: {SAM2_CHECKPOINT}")
else:
    print(f"✓ Checkpoint already exists: {SAM2_CHECKPOINT}")

# Verify download
print(f"✓ SAM2 base-plus checkpoint ready ({SAM2_CHECKPOINT.stat().st_size / 1e6:.1f} MB)")

## Train Improved RF-DETR


In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import sys

# Set up paths
HOME = Path.home()
PROJECT_DIR = Path("C:/Users/20228014/Desktop/final_project")
# Add RF-DETR and SAM2 to path
sys.path.insert(0, str(PROJECT_DIR / "rf-detr"))
sys.path.insert(0, str(PROJECT_DIR / "sam2"))

# Training configuration for crack detection with improved architecture
train_cmd = [
    sys.executable,
    "-m", "rfdetr.main",
    "--encoder", "dinov2_base",
    "--dataset_file", "coco",
    "--dataset_dir", "C:/Users/20228014/Desktop/crack-augmented",
    "--output_dir", "../outputfulal34enc0"
    "/crack_detection",
    "--batch_size", "2",
    "--grad_accum_steps","8",
    "--epochs", "70",
    "--lr", "5e-5",
    "--num_classes", "5",
    "--projector_scale","P3", "P4", "P5", "P6",
    "--enc_layers", "3",
    "--num_feature_levels", "4",
    "--dec_layers", "4",
    "--num_queries", "50",
    "--resolution", "512",               
    "--print_freq", "1",
    "--two_stage",
    "--device", "cuda",
    "--num_workers", "0",
    "--square_resize_div_64",
    "--eval_max_dets", "100"
    ]

print("⏳ Starting training...")
subprocess.run(train_cmd, cwd=str(PROJECT_DIR / "rf-detr"), check=True)
print("✓ Training completed!")


## Initialize SAM2

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Device configuration - CPU compatible
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Build SAM2 model (MANDATORY: use  checkpoint)
sam2_model = build_sam2(
    SAM2_CONFIG, 
    str(SAM2_CHECKPOINT), 
    device=DEVICE, 
    apply_postprocessing=False
)

# Create predictor
predictor = SAM2ImagePredictor(sam2_model)

print(f"✓ SAM2  initialized on {DEVICE}")
print(f"✓ Config: {SAM2_CONFIG}")
print(f"✓ Checkpoint: {SAM2_CHECKPOINT.name}")

## Run Complete Pipeline


In [ ]:
# Use the pipeline script we created
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__3044_jpg.rf.d0cde04c38b29a4bcedecbe0fd97b9bb.jpg"
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__3050_jpg.rf.50ed7a60a6e5203c2b727d7b599a06b1.jpg'
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__3117_jpg.rf.2fca06a3d3e8c450439a29b7890cf549.jpg"
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__3023_jpg.rf.693b88119430fc1386028eddfed817a5.jpg"
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-6__0511_4_jpg.rf.5c9609f934f18327dddfb0bb1251a0e4.jpg
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__3326_jpg.rf.9bba32083350f7ec042e3921fdf53379.jpg
#/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-5__2094_jpg.rf.e9924e9e2698ff9b304be98fe16226eb.jpg
#TEST_IMAGE="/Users/admin/Desktop/RF-DETR&SAM2/dataset/test/crack-6__0723-2_jpg.rf.a3d64bf66405c63ca76bb572e466a798.jpg"
RFDETR_CHECKPOINT = "/Users/admin/Desktop/fullnop2_34/checkpoint_best_total.pth"
OUTPUT_PATH = PROJECT_DIR / "result_crack_segmentation.jpg"

# Run pipeline using post-processing chain before SAM2
!python run_crack_pipeline.py \
    --image {TEST_IMAGE} \
    --rfdetr-checkpoint {RFDETR_CHECKPOINT} \
    --sam2-checkpoint {SAM2_CHECKPOINT} \
    --sam2-config {SAM2_CONFIG} \
    --output {OUTPUT_PATH} \
    --confidence 0.85\
    --nms-iou 0.1 \
    --num-positive-points 6\
    --num-negative-points 3 \
    --prompt-mode box+points \

print(f"\n✓ Pipeline completed!")
print(f"✓ Output saved: {OUTPUT_PATH}")

## Visualize 


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Display the result
result_img = Image.open(OUTPUT_PATH)

plt.figure(figsize=(15, 5))
plt.imshow(result_img)
plt.axis('off')
plt.title('Crack Detection + SAM2 Segmentation Result', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Complete pipeline executed successfully!")

In [ ]:
import json
import cv2
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from pycocotools.coco import COCO
from scipy.ndimage import binary_dilation
from run_crack_pipeline import CrackDetectionPipeline


DATASET_DIR = Path(f"/Users/admin/Desktop/dataset/batches/batch_{i}")
ANNOTATION_FILE = DATASET_DIR / "_annotations.coco.json"
RFDETR_CHECKPOINT = "/Users/admin/Desktop/fullnop2_34/checkpoint_best_total.pth"
SAM2_CHECKPOINT   = "/Users/admin/Desktop/final_project/checkpoints/sam2_hiera_base_plus.pt"
DEVICE            = "cpu"
CONFIDENCE        = 0.9
NMS_IOU           = 0.1
NUM_DARK_POINT    = 6
NUM_NEGATIVE_POINT= 3
PROMPT_MODES      = "box+points"
MASK_TOL          = 4   # tolerance in pixels for tolerant metrics

# ================================================
#  METRIC FUNCTIONS
# ================================================

# ---------------- HELPER ---------------- #

def resize_to_gt(pred_mask, gt_mask):
    return cv2.resize(
        pred_mask.astype(np.uint8),
        (gt_mask.shape[1], gt_mask.shape[0]),
        interpolation=cv2.INTER_NEAREST
    ).astype(bool)


# ---------------- 1. DICE ---------------- #

def compute_dice(mask1, mask2):
    """
    Standard Dice (no tolerance).
    Dice = 2*|A∩B| / (|A| + |B|)
    """
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)
    intersection = np.logical_and(mask1, mask2).sum()
    return (2. * intersection) / (mask1.sum() + mask2.sum() + 1e-6)


# ---------------- 2. TOLERANT DICE ---------------- #

def compute_tolerant_dice(mask1, mask2, tol=4):
    """
    Tolerant Dice.
    A pred pixel is accepted if within `tol` pixels of GT,
    and a GT pixel is forgiven if pred is within `tol` pixels of it.

    mask1 = prediction
    mask2 = ground truth
    """
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    mask1_d = binary_dilation(mask1, iterations=tol)  # pred tolerance zone
    mask2_d = binary_dilation(mask2, iterations=tol)  # GT  tolerance zone

    tp = np.logical_and(mask1,  mask2_d).sum()   # pred pixels inside GT zone
    fp = np.logical_and(mask1, ~mask2_d).sum()   # pred pixels outside GT zone
    fn = np.logical_and(mask2, ~mask1_d).sum()   # GT pixels outside pred zone

    return (2. * tp) / (2. * tp + fp + fn + 1e-6)


# ---------------- 3. MASK IoU ---------------- #

def compute_mask_iou(mask1, mask2):
    """
    Standard Mask IoU (no tolerance).
    IoU = |A∩B| / |A∪B|
    Always <= Dice.  Relationship: IoU = Dice / (2 - Dice)
    """
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)
    intersection = np.logical_and(mask1, mask2).sum()
    union        = np.logical_or(mask1,  mask2).sum()
    return intersection / (union + 1e-6)


# ---------------- 4. TOLERANT MASK IoU ---------------- #

def compute_tolerant_mask_iou(mask1, mask2, tol=4):
    """
    Tolerant Mask IoU.
    Same tolerance logic as tolerant Dice but computes IoU:
    IoU = TP / (TP + FP + FN)

    mask1 = prediction
    mask2 = ground truth
    """
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    mask1_d = binary_dilation(mask1, iterations=tol)
    mask2_d = binary_dilation(mask2, iterations=tol)

    tp = np.logical_and(mask1,  mask2_d).sum()
    fp = np.logical_and(mask1, ~mask2_d).sum()
    fn = np.logical_and(mask2, ~mask1_d).sum()

    return tp / (tp + fp + fn + 1e-6)


# ---------------- 5. BOX IoU ---------------- #

def compute_box_iou(box1, box2):
    """
    Standard Box IoU.
    box format: [x, y, w, h]  (COCO format)
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    inter_x1   = max(x1, x2)
    inter_y1   = max(y1, y2)
    inter_x2   = min(x1 + w1, x2 + w2)
    inter_y2   = min(y1 + h1, y2 + h2)

    inter_w    = max(0.0, inter_x2 - inter_x1)
    inter_h    = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    union_area = (w1 * h1) + (w2 * h2) - inter_area

    return inter_area / (union_area + 1e-6)


# ================================================
#  MATCHING + EVALUATION
# ================================================

def match_and_evaluate(
    gt_boxes,
    gt_masks,
    pred_boxes,
    pred_masks,
    iou_thresh_box=0.5,
    mask_tol=MASK_TOL
):
    matched_gt   = set()
    metrics_list = []

    for i, p_box in enumerate(pred_boxes):
        p_coco = [p_box[0], p_box[1], p_box[2] - p_box[0], p_box[3] - p_box[1]]
        p_mask = pred_masks[i]

        best_iou = 0
        best_j   = -1

        for j, g_box in enumerate(gt_boxes):
            if j in matched_gt:
                continue
            iou = compute_box_iou(g_box, p_coco)
            if iou > best_iou:
                best_iou = iou
                best_j   = j

        if best_j != -1:
            g_mask    = gt_masks[best_j]
            p_resized = resize_to_gt(p_mask, g_mask)

            # ── Compute all 5 metrics ──────────────────────────
            dice       = compute_dice(p_resized, g_mask)               # 1
            t_dice     = compute_tolerant_dice(p_resized, g_mask, tol=mask_tol)  # 2
            mask_iou   = compute_mask_iou(p_resized, g_mask)           # 3
            t_mask_iou = compute_tolerant_mask_iou(p_resized, g_mask, tol=mask_tol)  # 4
            # box_iou = best_iou (already computed above)              # 5
            # ──────────────────────────────────────────────────

            metrics_list.append({
                "box_iou"   : best_iou,
                "dice"      : dice,
                "t_dice"    : t_dice,
                "mask_iou"  : mask_iou,
                "t_mask_iou": t_mask_iou,
                "is_tp"     : best_iou >= iou_thresh_box
            })

            if best_iou >= iou_thresh_box:
                matched_gt.add(best_j)

    # ---- Detection counts ----
    TP = sum(m["is_tp"] for m in metrics_list)
    FP = len(pred_boxes) - TP
    FN = len(gt_boxes)   - TP

    # ---- Helper: average a metric over all matches or TP only ----
    def avg(key, tp_only=False):
        vals = [m[key] for m in metrics_list if (m["is_tp"] if tp_only else True)]
        return float(np.mean(vals)) if vals else 0.0

    return {
        # Detection
        "TP"           : TP,
        "FP"           : FP,
        "FN"           : FN,
        "precision"    : TP / (TP + FP + 1e-6),
        "recall"       : TP / (TP + FN + 1e-6),
        "f1"           : (2 * TP) / (2 * TP + FP + FN + 1e-6),

        # 1. Box IoU  (TP only — standard practice)
        "box_iou"      : avg("box_iou",    tp_only=True),

        # 2. Dice  (all matches / TP only)
        "dice"         : avg("dice"),
        "dice_tp"      : avg("dice",        tp_only=True),

        # 3. Tolerant Dice  (all matches / TP only)
        "t_dice"       : avg("t_dice"),
        "t_dice_tp"    : avg("t_dice",      tp_only=True),

        # 4. Mask IoU  (all matches / TP only)
        "mask_iou"     : avg("mask_iou"),
        "mask_iou_tp"  : avg("mask_iou",    tp_only=True),

        # 5. Tolerant Mask IoU  (all matches / TP only)
        "t_mask_iou"   : avg("t_mask_iou"),
        "t_mask_iou_tp": avg("t_mask_iou",  tp_only=True),
    }


# ================================================
#  MAIN EXECUTION
# ================================================

coco_gt   = COCO(str(ANNOTATION_FILE))
image_ids = coco_gt.getImgIds()

pipeline = CrackDetectionPipeline(
    RFDETR_CHECKPOINT,
    str(SAM2_CHECKPOINT),
    "sam2_hiera_b+.yaml",
    DEVICE
)

per_image_results = []

for img_id in tqdm(image_ids):
    img_info = coco_gt.loadImgs(img_id)[0]
    img_path = DATASET_DIR / img_info["file_name"]

    ann_ids  = coco_gt.getAnnIds(imgIds=img_id)
    anns     = coco_gt.loadAnns(ann_ids)

    gt_boxes = [a["bbox"] for a in anns]
    gt_masks = [coco_gt.annToMask(a) for a in anns]

    result = pipeline.run_pipeline(
        image_path=str(img_path),
        confidence_threshold=CONFIDENCE,
        nms_iou_threshold=NMS_IOU,
        output_path=None,
        num_dark_points=NUM_DARK_POINT,
        num_negative_points=NUM_NEGATIVE_POINT,
        max_cracks=3,
        prompt_mode=PROMPT_MODES
    )

    if result is None:
        metrics = {
            "TP": 0, "FP": 0, "FN": len(gt_boxes),
            "precision"    : 0, "recall": 0, "f1": 0,
            "box_iou"      : 0,
            "dice"         : 0, "dice_tp"      : 0,
            "t_dice"       : 0, "t_dice_tp"    : 0,
            "mask_iou"     : 0, "mask_iou_tp"  : 0,
            "t_mask_iou"   : 0, "t_mask_iou_tp": 0,
        }
    else:
        metrics = match_and_evaluate(
            gt_boxes,
            gt_masks,
            result["selected_boxes"],
            result["masks"]
        )

    per_image_results.append({
        "image_id" : img_id,
        "file_name": img_info["file_name"],
        **metrics
    })

df = pd.DataFrame(per_image_results)
df.to_csv("crack_metrics.csv", index=False)
df.to_csv(f'/Users/admin/Desktop/dataset/merged-cracks/batches/NoP2_85/batch_{i}.csv', index=False)
# ================================================
#  FINAL SUMMARY
# ================================================


print("\n================ FINAL SUMMARY ================")

print(f"\n--- Detection ---")
print(f"  Precision  : {df['precision'].mean():.4f}")
print(f"  Recall     : {df['recall'].mean():.4f}")
print(f"  F1         : {df['f1'].mean():.4f}")



print(f"\n--- 2. Dice              (all | ) ---")
print(f"  Dice       : {df['dice'].mean():.4f} ")

print(f"\n--- 3. Tolerant Dice tol={MASK_TOL}px  (all ) ---")
print(f"  T-Dice     : {df['t_dice'].mean():.4f} ")

print(f"\n--- 4. Mask IoU          (all) ---")
print(f"  Mask IoU   : {df['mask_iou'].mean():.4f} ")

print(f"\n--- 5. Tolerant Mask IoU tol={MASK_TOL}px  (all ")
print(f"  T-Mask IoU : {df['t_mask_iou'].mean():.4f}")

print(f"\n--- Sanity check: Dice → IoU via formula ---")
d  = df['dice'].mean()
td = df['t_dice'].mean()
print(f"  Standard : Dice={d:.4f}  formula IoU={d/(2-d):.4f}  actual IoU={df['mask_iou'].mean():.4f}")
print(f"  Tolerant : Dice={td:.4f} formula IoU={td/(2-td):.4f} actual IoU={df['t_mask_iou'].mean():.4f}")

In [4]:
df.describe()

,image_id,TP,FP,FN,precision,recall,f1,box_iou,dice,dice_tp,t_dice,t_dice_tp,mask_iou,mask_iou_tp,t_mask_iou,t_mask_iou_tp
count,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000,426.000000
mean,588.464789,1.058685,0.140845,0.492958,0.908059,0.821018,0.842687,0.829247,0.689537,0.679337,0.833092,0.818322,0.565713,0.560354,0.759905,0.751420
std,167.644245,0.453828,0.438042,1.049906,0.258877,0.312988,0.284387,0.232421,0.218820,0.246644,0.213574,0.256361,0.229796,0.246422,0.244058,0.273729
min,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,503.250000,1.000000,0.000000,0.000000,0.999999,0.616667,0.666667,0.823394,0.579842,0.580983,0.771901,0.776983,0.414103,0.412140,0.639532,0.639946
50%,609.500000,1.000000,0.000000,0.000000,0.999999,0.999999,1.000000,0.906049,0.733271,0.745411,0.909723,0.920769,0.580706,0.595174,0.834771,0.856704
75%,715.750000,1.000000,0.000000,1.000000,0.999999,0.999999,1.000000,0.956534,0.857151,0.858829,0.976530,0.979539,0.750016,0.752586,0.954136,0.959899
max,822.000000,3.000000,3.000000,8.000000,1.000000,1.000000,1.000000,0.993045,0.986244,0.986244,1.000000,1.000000,0.972861,0.972861,1.000000,1.000000
